# Binary Motif Features for Time-Series Classification

This tutorial turns class-discriminative subsequences into binary features. It uses the GunPoint dataset and the matrix-profile approach from the [Shapelet Discovery tutorial](Tutorial_Shapelet_Discovery.ipynb). Each candidate pattern becomes one feature: a value of `1` means a close match occurs somewhere in that sample. A sample can contain several patterns, so its feature vector is multi-hot.

Pattern discovery and distance thresholds use only the training split. The test split is used only for the final evaluation. The candidates below are shapelet-like motif prototypes; the same encoding can be applied to motifs discovered by another method.

In [ ]:
import numpy as np
import pandas as pd
import stumpy
from sklearn import metrics, tree

## Load the labeled series

Each row is one time series. The first column contains the class (`0` for `Gun`, `1` for `Point`).

In [ ]:
train_url = "https://zenodo.org/record/4281349/files/gun_point_train_data.csv?download=1"
test_url = "https://zenodo.org/record/4281349/files/gun_point_test_data.csv?download=1"
train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

y_train = train_df.iloc[:, 0].to_numpy()
y_test = test_df.iloc[:, 0].to_numpy()
X_train = train_df.iloc[:, 1:].to_numpy(dtype=float)
X_test = test_df.iloc[:, 1:].to_numpy(dtype=float)

## Find candidate patterns from the training classes

Concatenating samples with a `NaN` separator lets a matrix profile compare subsequences from different samples without creating matches across sample boundaries. The profile contrast favors patterns that are conserved within one class and less similar to the other class. We select non-overlapping high-contrast candidates from each class.

In [ ]:
def concatenate_samples(samples):
    return np.concatenate([np.r_[sample, np.nan] for sample in samples])


def discover_candidates(source, other, m, k=5):
    P_self = stumpy.stump(source, m)[:, 0].astype(float)
    P_other = stumpy.stump(source, m, other, ignore_trivial=False)[:, 0].astype(float)
    contrast = P_other - P_self
    contrast[~np.isfinite(contrast)] = -np.inf

    selected = []
    for idx in np.argsort(contrast)[::-1]:
        if not np.isfinite(contrast[idx]):
            break
        if all(abs(idx - previous) >= m for previous in selected):
            selected.append(int(idx))
        if len(selected) == k:
            break

    return [source[idx : idx + m].copy() for idx in selected]


m = 38
gun_series = concatenate_samples(X_train[y_train == 0])
point_series = concatenate_samples(X_train[y_train == 1])

motifs = []
motif_classes = []
for label, source, other in [(0, gun_series, point_series), (1, point_series, gun_series)]:
    candidates = discover_candidates(source, other, m)
    motifs.extend(candidates)
    motif_classes.extend([label] * len(candidates))

print(f"Discovered {len(motifs)} candidate patterns")

## Convert motif distances to binary features

For each candidate, compute its minimum MASS distance to every sample. We set that candidate's presence threshold to the 95th percentile of distances among training samples from the class where the candidate was discovered. The rule is then fixed and applied to both splits. Thresholds tuned this way avoid using test samples to define what counts as a match.

In [ ]:
def motif_distances(samples, motifs):
    distances = np.empty((len(samples), len(motifs)))
    for j, motif in enumerate(motifs):
        for i, sample in enumerate(samples):
            distances[i, j] = stumpy.mass(motif, sample).min()
    return distances


D_train = motif_distances(X_train, motifs)
D_test = motif_distances(X_test, motifs)
motif_classes = np.asarray(motif_classes)
same_class = y_train[:, None] == motif_classes[None, :]
thresholds = np.array([
    np.percentile(D_train[same_class[:, j], j], 95)
    for j in range(len(motifs))
])

X_train_features = (D_train <= thresholds[None, :]).astype(np.uint8)
X_test_features = (D_test <= thresholds[None, :]).astype(np.uint8)
print("Feature matrix shape:", X_train_features.shape)

## Train and evaluate a classifier

The decision tree receives one binary column per candidate pattern. Its predictions are evaluated once on the held-out test samples. Different data may need a different motif length, number of candidates, or threshold; choose these using training data or cross-validation.

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=3, random_state=0)
clf.fit(X_train_features, y_train)
y_pred = clf.predict(X_test_features)
print(f"Test accuracy = {metrics.accuracy_score(y_test, y_pred):.3f}")

## Summary

Matrix-profile contrasts can provide candidate class-associated patterns. Measuring their minimum distance in each sample and applying training-derived thresholds produces one-hot or multi-hot features that can be passed to standard supervised models.